# Exercise 1 — Build a RAG Q&A System (Google Colab)

### The story
You are building a **research assistant for your PhD defense**. You will feed it a
document (we use a short story PDF in the workshop), and it will answer questions
using *only* what's inside that document. This is **RAG** — Retrieval-Augmented
Generation.

### What you'll build in this notebook
```
PDF  →  split into chunks  →  embed  →  store in Pinecone  →  retrieve  →  ask the LLM
```

### Before you start
1. **Pinecone** API key (free Starter plan)
2. **LLM key** — add **one or both** in Colab secrets (🔑):
   - `GOOGLE_API_KEY` — https://aistudio.google.com/apikey (free, default)
   - `OPENAI_API_KEY` — https://platform.openai.com/api-keys (paid)
3. Always add: `PINECONE_API_KEY`

> **Stack:** Pinecone `story-llama` + integrated embed. **LLM:** set `LLM_PROVIDER` in Step 1 (`"google"` or `"openai"`).


## Step 0 — Install the libraries
Run this cell first. Use **`%pip`** (not `!pip`) so packages install into the active kernel.
After it finishes, if you already ran other cells in an old session, use **Runtime → Restart session**
and run from Step 0 again.


In [ ]:
%pip install -q pinecone pypdf langchain-community langchain-text-splitters langchain langchain-core langchain-google-genai google-generativeai langchain-openai openai

## Step 1 — Load API keys & pick LLM provider

1. Colab **Secrets** (🔑): add `PINECONE_API_KEY` plus **either** `GOOGLE_API_KEY` or `OPENAI_API_KEY` (both is fine).
2. Set `LLM_PROVIDER` below to match the key you want to use.


In [ ]:
LLM_PROVIDER = "google"  # "google" | "openai"

from google.colab import userdata
import os

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

if LLM_PROVIDER == "google":
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
elif LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    raise ValueError('LLM_PROVIDER must be "google" or "openai"')

print(f"Keys loaded · LLM provider: {LLM_PROVIDER}")

## Step 2 — Upload your document
Run the cell, then click **Choose Files** and pick your PDF (the workshop story, or
any PDF you like). It uploads into Colab's `/content/` folder.


In [ ]:
from google.colab import files

uploaded = files.upload()
# select your PDF when prompted — it uploads to /content/ automatically

pdf_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_filename}")

## Step 3 — Ingest: load → split → embed → store
This is the heart of RAG ingestion. Four things happen:

1. **Load** the PDF into text (one document per page).
2. **Split** the text into ~500-character chunks so retrieval is precise.
3. **Embed** each chunk with Pinecone's free **`llama-text-embed-v2`** model (no OpenAI call).
4. **Store** the chunks in your **`story-llama`** Pinecone index.

**What to watch for:** the printed page count and chunk count. More pages → more chunks.

**Checkpoint:** you should see `Done! PDF indexed into Pinecone successfully.`

### 🔧 Your turn (experiment)
After it works once, try changing `chunk_size=500` to `chunk_size=1000` and re-run.
Fewer, bigger chunks = more context per hit but less precise retrieval. Notice how the
chunk count drops.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone

INDEX_NAME, NAMESPACE = "story-llama", "default"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
if not pc.has_index(INDEX_NAME):
    pc.create_index_for_model(
        name=INDEX_NAME, cloud="aws", region="us-east-1",
        embed={"model": "llama-text-embed-v2", "field_map": {"text": "chunk_text"}},
    )
index = pc.Index(INDEX_NAME)

docs = PyPDFLoader(pdf_filename).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
print(f"Loaded: {len(docs)} pages · Chunks: {len(chunks)}")

index.upsert_records(namespace=NAMESPACE, records=[
    {"id": f"chunk-{i}", "chunk_text": c.page_content} for i, c in enumerate(chunks)
])
print("Done! PDF indexed into Pinecone successfully.")

## Step 3b — Test retrieval (Pinecone only)
Run this to confirm your chunks are searchable **before** wiring up the LLM.
You should see the top 3 matching text snippets from your PDF.

In [ ]:
test_query = "What is the story about?"
hits = index.search(
    namespace=NAMESPACE,
    query={"inputs": {"text": test_query}, "top_k": 3},
    fields=["chunk_text"],
)["result"]["hits"]

print(f"Query: {test_query} · Found {len(hits)} matches\n")
for i, hit in enumerate(hits, 1):
    text = hit["fields"]["chunk_text"]
    print(f"--- Match {i} ---\n{text[:300]}{'...' if len(text) > 300 else ''}\n")

## Step 4 — Build the retrieval chain

1. **Retriever** — Pinecone search, top `k=4` chunks (same for both providers).
2. **LLM** — Gemini or OpenAI per `LLM_PROVIDER` from Step 1.
3. **Prompt** — answer from context only; say "I don't know" otherwise.
4. **`chat()`** — runs the chain with short memory.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

def get_llm():
    if LLM_PROVIDER == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

def retrieve(question, top_k=4):
    hits = index.search(
        namespace=NAMESPACE,
        query={"inputs": {"text": question}, "top_k": top_k},
        fields=["chunk_text"],
    )["result"]["hits"]
    return [h["fields"]["chunk_text"] for h in hits]

def to_text(content) -> str:
    if isinstance(content, str): return content
    if isinstance(content, dict): return content.get("text", str(content))
    if isinstance(content, list): return "".join(to_text(p) for p in content)
    return str(content)

llm = get_llm()
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using the context below. If you don't know, say you don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])
chat_history = []
chain = (
    {"context": lambda x: "\n\n".join(retrieve(x["question"])),
     "chat_history": lambda x: x["chat_history"],
     "question": lambda x: x["question"]}
    | prompt | llm | StrOutputParser()
)

def chat(question):
    global chat_history
    answer = to_text(chain.invoke({"question": question, "chat_history": chat_history}))
    chat_history += [HumanMessage(content=question), AIMessage(content=answer)]
    chat_history = chat_history[-8:]
    return answer

## Step 5 — Ask questions
Run this to see your RAG system answer three questions about the document.

### 🔧 Your turn
Edit the `questions` list — add a question you *know* the answer to from the document,
and one that is **not** in the document. Confirm the second one gets an "I don't know"
style answer. That proves the model is grounded and not making things up.


In [ ]:
questions = [
    "What is the story about?",
    "Who are the main characters?",
    "What is the moral of the story?"
]

for q in questions:
    answer = chat(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## ✅ Done
Pinecone `story-llama` RAG chain with your chosen LLM provider. Next: Exercise 2 turns retrieval into an agent tool.
